In [1]:
import os, re, glob
import numpy as np, pandas as pd, pydicom

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
tr = pd.read_csv(f"{COMP}/train.csv")
ser_tr = pd.read_csv(f"{COMP}/train_series.csv")
ser_ts = pd.read_csv(f"{COMP}/test_series.csv")

def read_header(split, study, series):
    files = sorted(glob.glob(f"{COMP}/{split}_series/{study}/{series}/*.dcm"))
    if not files:
        return None
    d = pydicom.dcmread(files[0], stop_before_pixels=True)
    g = lambda k: str(getattr(d, k, "") or "").strip()
    row = {k: g(k) for k in ["Laterality", "ImageLaterality", "BodyPartExamined", "SeriesDescription",
                             "ProtocolName", "StudyDescription", "PatientPosition", "Manufacturer"]}
    try:  # titik tengah citra dalam koordinat pasien (LPS: x positif = sisi KIRI pasien)
        ipp = np.array(d.ImagePositionPatient, float); iop = np.array(d.ImageOrientationPatient, float)
        ps = np.array(d.PixelSpacing, float)
        c = ipp + iop[:3] * ps[1] * d.Columns / 2 + iop[3:] * ps[0] * d.Rows / 2
        row["center_x"] = float(c[0])
    except Exception:
        row["center_x"] = np.nan
    return row

studies = tr["StudyInstanceUID"].sample(300, random_state=0).tolist()
rows = []
for split, ser, uids in [("train", ser_tr, studies), ("test", ser_ts, ser_ts["StudyInstanceUID"].unique().tolist())]:
    for uid in uids:
        for _, s in ser[ser["StudyInstanceUID"] == uid].iterrows():
            h = read_header(split, uid, s["SeriesInstanceUID"])
            if h:
                h.update(split=split, StudyInstanceUID=uid, plane=s["Anatomical_Plane"])
                rows.append(h)
H = pd.DataFrame(rows)
print("Header terbaca:", len(H), "series dari", H["StudyInstanceUID"].nunique(), "studi")

Header terbaca: 1687 series dari 303 studi


In [2]:
print("=== Proporsi series yang tag-nya TERISI ===")
for k in ["Laterality", "ImageLaterality", "BodyPartExamined", "SeriesDescription", "ProtocolName", "StudyDescription"]:
    print(f"{k:18s} {(H[k] != '').mean():.0%}")

print("\n=== Isi tag Laterality / ImageLaterality ===")
print(H["Laterality"].value_counts().head(), "\n", H["ImageLaterality"].value_counts().head())

# kata kiri/kanan di teks deskripsi (berbagai bahasa)
LEFT = r"\b(L|LT|LEFT|LINKS|LI|IZQ|IZQUIERDA|IZQUIERDO|GAUCHE|SX|SIN|SINISTRO|SINISTRA|ESQ|ESQUERDO|ESQUERDA)\b"
RIGHT = r"\b(R|RT|RIGHT|RECHTS|RE|DER|DERECHA|DERECHO|DROITE|DX|DEX|DESTRO|DESTRA|DIR|DIREITO|DIREITA)\b"
def side_from_text(t):
    t = re.sub(r"[^A-Z]+", " ", t.upper())
    l, r = bool(re.search(LEFT, t)), bool(re.search(RIGHT, t))
    return "kiri" if l and not r else ("kanan" if r and not l else "")
H["txt"] = H["SeriesDescription"] + " " + H["ProtocolName"] + " " + H["StudyDescription"] + " " + H["BodyPartExamined"]
H["side_tag"] = H["Laterality"].str.upper().map({"L": "kiri", "R": "kanan"}).fillna("")
H["side_txt"] = H["txt"].apply(side_from_text)

per_study = H.groupby("StudyInstanceUID").agg(
    split=("split", "first"),
    tag=("side_tag", lambda s: next((v for v in s if v), "")),
    txt=("side_txt", lambda s: next((v for v in s if v), "")),
    cx=("center_x", lambda s: np.nanmedian(s.values) if s.notna().any() else np.nan))
per_study["geo"] = np.where(per_study["cx"] > 0, "kiri", np.where(per_study["cx"] < 0, "kanan", ""))
print("\n=== Studi yang punya info kiri/kanan ===")
for src in ["tag", "txt", "geo"]:
    print(f"{src:4s} -> latih {(per_study.loc[per_study.split=='train', src] != '').mean():.0%} | "
          f"uji {(per_study.loc[per_study.split=='test', src] != '').mean():.0%}")
print("\nContoh teks deskripsi:"); print(H["txt"].drop_duplicates().head(15).to_string())

=== Proporsi series yang tag-nya TERISI ===
Laterality         47%
ImageLaterality    0%
BodyPartExamined   93%
SeriesDescription  95%
ProtocolName       0%
StudyDescription   0%

=== Isi tag Laterality / ImageLaterality ===
Laterality
     893
R    453
L    341
Name: count, dtype: int64 
 ImageLaterality
    1687
Name: count, dtype: int64

=== Studi yang punya info kiri/kanan ===
tag  -> latih 47% | uji 33%
txt  -> latih 20% | uji 0%
geo  -> latih 100% | uji 100%

Contoh teks deskripsi:
0                pd_tse_fs_sag_np   
1                pd_tse_fs_tra_np   
2                     t2_me2d_sag   
3                pd_tse_fs_cor_np   
4                   t1_se_cor_384   
5             R- Ax PD FSE FS   KNEE
6             R- CORPD FSE FS   KNEE
7                R Cor T1 FSE   KNEE
8                R SAG T1 FSE   KNEE
9            R- Sag PD FSE FS   KNEE
10    pd_tse_fs_cor_384_p2_s2   KNEE
11    pd_tse_fs_tra_448_p2_s2   KNEE
12       t1_tse_cor_448_p2_s2   KNEE
13           pd_tse_fs_sag

In [3]:
REP_L = r"\b(left|izquierd\w*|links|gauche|sinistr\w*|esquerd\w*|lewe|lewy|vänster|venstre|vasen|izq)\b"
REP_R = r"\b(right|derech\w*|rechts|droit\w*|destr\w*|direit\w*|prawe|prawy|höger|høyre|oikea|der)\b"
def side_from_report(t):
    t = str(t).lower()
    l, r = len(re.findall(REP_L, t)), len(re.findall(REP_R, t))
    return "kiri" if l > r and l > 0 else ("kanan" if r > l else "")
rep = tr.set_index("StudyInstanceUID")["Report"]
lat = per_study[per_study.split == "train"].copy()
lat["laporan"] = [side_from_report(rep.get(u, "")) for u in lat.index]
known = lat[lat["laporan"] != ""]
print(f"Studi latih dengan sisi jelas di laporan: {len(known)} dari {len(lat)}")
print("Sebaran sisi dari laporan:", known["laporan"].value_counts().to_dict())
for src in ["tag", "txt", "geo"]:
    k = known[known[src] != ""]
    if len(k):
        print(f"{src:4s}: tersedia {len(k):3d} studi | cocok dengan laporan {(k[src] == k['laporan']).mean():.0%}")
    else:
        print(f"{src:4s}: tidak tersedia")
print("\nGeometri vs laporan (posisi x tengah citra, mm):")
print(known.groupby("laporan")["cx"].describe()[["count", "mean", "50%", "min", "max"]].round(1))

Studi latih dengan sisi jelas di laporan: 112 dari 300
Sebaran sisi dari laporan: {'kanan': 77, 'kiri': 35}
tag : tersedia  62 studi | cocok dengan laporan 77%
txt : tersedia  29 studi | cocok dengan laporan 100%
geo : tersedia 112 studi | cocok dengan laporan 88%

Geometri vs laporan (posisi x tengah citra, mm):
         count  mean   50%    min    max
laporan                                 
kanan     77.0 -51.3 -76.8 -111.9  102.4
kiri      35.0  83.1  82.2   16.9  122.4


In [4]:
# (1) Acuan laporan yang lebih bersih: buang 'der' (kata sandang Jerman), hanya kata sisi yang jelas
REP_L2 = r"\b(left|izquierd[ao]|linke[nrs]?|links|gauche|sinistr[oa]|esquerd[oa]|lewe|lewy|vänster|venstre)\b"
REP_R2 = r"\b(right|derech[ao]|rechte[nrs]?|rechts|droite?|destr[oa]|direit[oa]|prawe|prawy|höger|høyre)\b"
def side2(t):
    t = str(t).lower(); l, r = len(re.findall(REP_L2, t)), len(re.findall(REP_R2, t))
    return "kiri" if l > r else ("kanan" if r > l else "")
lat["laporan2"] = [side2(rep.get(u, "")) for u in lat.index]
k2 = lat[lat["laporan2"] != ""]
print(f"Acuan laporan (bersih): {len(k2)} studi | {k2['laporan2'].value_counts().to_dict()}")
for thr in [0, 20, 40]:
    k = k2[k2["cx"].abs() > thr]
    print(f"geo vs laporan, |x| > {thr:2d} mm: {len(k):3d} studi | cocok {(k['geo'] == k['laporan2']).mean():.0%}")

# (2) Kecocokan antar sumber DICOM sendiri (tanpa laporan)
for a, b in [("geo", "txt"), ("geo", "tag"), ("tag", "txt")]:
    k = per_study[(per_study[a] != "") & (per_study[b] != "")]
    print(f"{a} vs {b}: {len(k):3d} studi | cocok {(k[a] == k[b]).mean():.0%}")

# (3) Arah kolom gambar: apakah orientasi kiri-kanan sudah konsisten antar series?
ori = []
for split, ser, uids in [("train", ser_tr, studies)]:
    for uid in uids[:150]:
        for _, s in ser[ser["StudyInstanceUID"] == uid].iterrows():
            f = sorted(glob.glob(f"{COMP}/{split}_series/{uid}/{s['SeriesInstanceUID']}/*.dcm"))
            if not f: continue
            d = pydicom.dcmread(f[0], stop_before_pixels=True)
            try:
                iop = np.array(d.ImageOrientationPatient, float)
            except Exception:
                continue
            ori.append({"plane": s["Anatomical_Plane"], "row_x": round(iop[0], 1), "col_x": round(iop[3], 1),
                        "manufacturer": str(getattr(d, "Manufacturer", ""))[:20]})
O = pd.DataFrame(ori)
print("\nArah baris gambar (komponen x dari ImageOrientationPatient):")
print("+1 = kolom kanan gambar ke arah KIRI pasien, -1 = sebaliknya, 0 = sumbu kiri-kanan tidak di arah baris")
print(O.groupby("plane")["row_x"].apply(lambda s: s.round().value_counts().to_dict()).to_string())
print("\nKoronal/aksial yang arah barisnya -1, per pabrikan:")
ca = O[O["plane"].isin(["Coronal", "Axial"])]
print(ca.assign(neg=ca["row_x"] < -0.5).groupby("manufacturer")["neg"].agg(["mean", "count"]).round(2).to_string())

Acuan laporan (bersih): 93 studi | {'kanan': 54, 'kiri': 39}
geo vs laporan, |x| >  0 mm:  93 studi | cocok 96%
geo vs laporan, |x| > 20 mm:  89 studi | cocok 98%
geo vs laporan, |x| > 40 mm:  88 studi | cocok 98%
geo vs txt:  59 studi | cocok 100%
geo vs tag: 143 studi | cocok 96%
tag vs txt:  34 studi | cocok 94%

Arah baris gambar (komponen x dari ImageOrientationPatient):
+1 = kolom kanan gambar ke arah KIRI pasien, -1 = sebaliknya, 0 = sumbu kiri-kanan tidak di arah baris
plane        
Axial     1.0    217.0
          0.0      NaN
Coronal   1.0    294.0
          0.0      NaN
Sagittal  1.0      NaN
          0.0    327.0

Koronal/aksial yang arah barisnya -1, per pabrikan:
                      mean  count
manufacturer                     
CANON_MEC              0.0     21
GE MEDICAL SYSTEMS     0.0     80
GEHC                   0.0      3
Philips                0.0     82
Philips Medical Syst   0.0     88
SIEMENS                0.0    117
Siemens                0.0      6
Siemens